### Accidental and Deliberate Risks and Dependencies
#### Assigning each at the Sub-Sector level

In [2]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
#read the infrastructure merged dataset.
#should be an output that was named: 
hex_path = "\hex_infrastructure_merged.gpkg"
#Define GPD:
gdf_merged = gpd.read_file(hex_path)

### Highly Dependent Sectors - Both CISA and CL

In [ ]:
gdf_merged["Highly Dependent"] = (

gdf_merged["Broadband"]*11 +
gdf_merged["Cellular Towers"]*11 +

gdf_merged["Dam or Levee"]*2 +

gdf_merged["Fire Stations"] +
gdf_merged["Local Law Enforcement"] +

gdf_merged["Power Plants"]*9 +
gdf_merged["Electric Substations or Switching Stations"]*9 +

gdf_merged["Emergency Center"] +
gdf_merged["Hospitals"] +

gdf_merged["Strategic Highway Network"]*5 +

gdf_merged["Wastewater Treatment"]*6 +
gdf_merged["Water Point Data"]*6

).astype("int32")

#making the normalized 

min_val = gdf_merged["Highly Dependent"].min()
max_val = gdf_merged["Highly Dependent"].max()

gdf_merged["Highly Dependent Index"] = (
    (gdf_merged["Highly Dependent"] - min_val) / (max_val - min_val) * 100
).astype("int32")

# Check the result
gdf_merged[["Highly Dependent", "Highly Dependent Index"]].head()
gdf_merged["Highly Dependent Index"].describe()

In [ ]:
#simplifying the dataset
#trying to get the numbers right 
#arcgis commonly has trouble reading numbers outputted. need to coerce to int here, check types.

columns_to_keep = ["Highly Dependent", "Highly Dependent Index", "h3_ID", "geometry"]
gdf_subset = gdf_merged[columns_to_keep].copy()
gdf_subset = gdf_subset.rename(columns={
    "Highly Dependent": "Highly_Dependent",
    "Highly Dependent Index": "Highly_Dependent_Index"
})

for col in ["Highly_Dependent", "Highly_Dependent_Index"]:
    gdf_subset[col] = (
        pd.to_numeric(gdf_subset[col], errors="coerce")
        .fillna(0)
        .astype("int32")
    )


#need to check  !!!!!!!
print(gdf_subset.dtypes)


In [ ]:
gdf_merged.explore()

In [ ]:
gdf_subset.to_file(r"\hex_infrastructure_dependencies.gpkg", layer="data", driver="GPKG")

### Accidental and Deliberate Risk - CISA

In [ ]:
#Accidental
 #Airplane Crash
 #Cyber Incident
 #Industrial Accident
 #Power Failure
 #SCADA System Failure
 #Train Derailment
 #Urban Conflagration
#Deliberate
 #Armed Attack
 #Arson/Incendiary Attack
 #Civil Unrest
 #Conventional Bomb/Improvised Explosive
 #Cyber Incident
 #Sabotage
 #Theft

#Weights are Total Non Geographic Risks by Sector or Sub-Sector

weights = {
    "Chemical": 7,  
    "Communications": 4,
    "Emergency Services": 4,
    "Energy": 8,
    "Financial": 10,
    "Food & Ag": 7,
    "Government Facilities": 9,
    "Healthcare & Public Health": 10,
    "Transportation": 6,
    "Rail Lines": 1,
    "Rail Bridges": 1,
    "Rail Crossings": 1,
    "Water & Wastewater": 6,
}

#Treat any value in the hex bin that is 1 or greater as "present" abd add the corresponding weight value
gdf_merged["Non_Geo_Risk"] = sum(
    (gdf_merged[col].fillna(0) >= 1).astype(int) * weight
    for col, weight in weights.items()
    if col in gdf_merged.columns
)

#Make a column to normalize that risk
min_val = gdf_merged["Non_Geo_Risk"].min()
max_val = gdf_merged["Non_Geo_Risk"].max()

if max_val == min_val:
    gdf_merged["Non_Geo_Normalized"] = 100
else:
    gdf_merged["Non_Geo_Normalized"] = (
        (gdf_merged["Non_Geo_Risk"] - min_val) / (max_val - min_val)
    ) * 99 + 1

#Sanity Check
print(gdf_merged[["Non_Geo_Risk", "Non_Geo_Normalized"]].describe())

In [ ]:
#Save after this addition
#Save where you are putting edits to the H3 layer
gdf_merged.to_file(r"\hex_risk_non_geo.gpkg", layer="data", driver="GPKG")

### Accidental and Deliberate Risk - FEMA

In [ ]:
#read community lifelines data
hex_path = "\hex_community_lifelines.gpkg"
#Define GPD:
gdf_merged = gpd.read_file(hex_path)


#Commumity Lifeline Categories are Different - Review Crosswalk
#Weights are Total Non Geographic Risks by Sector or Sub-Sector
weights = {
    "Communications": 4,
    "Energy (Power & Fuel)": 8,
    "Food, Hydration, Shelter": 7,
    "Health and Medical": 10,
    "Safety and Security": 10,
    "Water Systems": 6,  
    "Transportation": 6,
    "Rail Lines": 1,
    "Rail Bridges": 1,
    "Rail Crossings": 1,
}

#Treat any value in the hex bin that is 1 or greater as "present" abd add the corresponding weight value
gdf_merged["Non_Geo_CL"] = sum(
    (gdf_merged[col].fillna(0) >= 1).astype(int) * weight
    for col, weight in weights.items()
    if col in gdf_merged.columns
)

#Make a column to normalize that risk
min_val = gdf_merged["Non_Geo_CL"].min()
max_val = gdf_merged["Non_Geo_CL"].max()

if max_val == min_val:
    gdf_merged["Non_Geo_CL_Norm"] = 100
else:
    gdf_merged["Non_Geo_CL_Norm"] = (
        (gdf_merged["Non_Geo_CL"] - min_val) / (max_val - min_val)
    ) * 99 + 1

#Sanity Check
print(gdf_merged[["Non_Geo_CL", "Non_Geo_CL_Norm"]].describe())

In [ ]:
#Save after this addition
#Save where you are putting edits to the H3 layer
gdf_merged.to_file(r"\hex_infrastructure_dependencies.gpkg", layer="data", driver="GPKG")